In [1]:
# !pip install dask-ml
# !pip install git+https://github.com/pycroscopy/sidpy.git@main
# !pip install "dask==2025.1.0" "distributed==2025.1.0"

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
import sidpy
import SciFiReaders as sr
import BGlib.be as belib
from BGlib.be.analysis.utils.sidpy_sho_fitter import SHOestimateGuess, SHO_fit_flattened
from sidpy.proc.fitter_refactor import SidpyFitterRefactor

In [3]:
%matplotlib widget

In [4]:
folder_path = r'../../../inputs/'
file_name = r'60x60_0deg_0004.h5'
path_to_file = os.path.join(folder_path, file_name)
patcher = belib.translators.LabViewH5Patcher()
patcher.translate(path_to_file)
reader = sr.Usid_reader(path_to_file)
data = reader.read()

C:\jawad_pc\2_code\20250912_code_pyc_bglib\env_pb2\lib\site-packages\sidpy\sid\translator.py:42: FutureWarning: Consider using sidpy.Reader instead of sidpy.Translator if possible and contribute your reader to ScopeReaders
  warn('Consider using sidpy.Reader instead of sidpy.Translator if '
2026-06-12 10:50:07,949 - BGlib.be.translators.labview_h5_patcher - INFO - File is already Pycroscopy ready.


In [5]:
data

sidpy.Dataset of type UNKNOWN with:
 dask.array<array, shape=(60, 60, 123, 64, 2, 2), dtype=complex64, chunksize=(45, 45, 45, 45, 2, 2), chunktype=numpy.ndarray>
 data contains: quantity (a.u.)
 and Dimensions: 
X:  X (nm) of size (60,)
Y:  Y (nm) of size (60,)
Frequency:  Frequency (Hz) of size (123,)
DC_Offset:  DC_Offset (V) of size (64,)
Field:  Field (generic) of size (2,)
Cycle:  Cycle (generic) of size (2,)

In [6]:
beps_raw = data
freq_axis = beps_raw.labels.index('Frequency (Hz)')
freq_vec = beps_raw._axes[freq_axis].values
all_dims = np.arange(len(data.shape))
ind_dims = np.delete(all_dims, freq_axis)
print(ind_dims, freq_vec.shape)

[0 1 3 4 5] (123,)


In [7]:
lb = [1E-6, freq_vec.min(), 50, -2*np.pi]
ub = [1E-3, freq_vec.max(), 500, 2*np.pi]
beps_small = beps_raw[:2, :2, :, :, :, :]
fitter = SidpyFitterRefactor(beps_small, SHO_fit_flattened, SHOestimateGuess, ind_dims=(freq_axis,), num_params=4, lower_bounds=lb, upper_bounds=ub)
fitter.setup_calc()

Setup Complete. Params: 4 | Spatial Dims: [0, 1, 3, 4, 5]


In [8]:
output = fitter.do_fit(use_kmeans=True, n_clusters=4, fit_parameter_labels=['Amplitude', 'Resonant Frequency', 'Quality Factor', 'Phase'])

C:\jawad_pc\2_code\20250912_code_pyc_bglib\env_pb2\lib\site-packages\sidpy\sid\dataset.py:1517: UserWarning: Dimensional information will be lost.                       Please use fold, unfold to combine dimensions
  warnings.warn('Dimensional information will be lost.\
2026-06-12 10:50:24,015 - root - INFO - Starting _check_array
2026-06-12 10:50:24,064 - root - INFO - Finished _check_array in 0:00:00.050186
2026-06-12 10:50:24,069 - root - INFO - Starting init_scalable
2026-06-12 10:50:24,071 - dask_ml.cluster.k_means - INFO - Initializing with k-means||
2026-06-12 10:50:24,141 - dask_ml.cluster.k_means - INFO - Starting init iteration  1/ 9 ,  1 centers


Starting Dask K-Means Guess with 4 clusters...


2026-06-12 10:50:24,213 - dask_ml.cluster.k_means - INFO - Finished init iteration  1/ 9 ,  1 centers in 0:00:00.071166
2026-06-12 10:50:24,238 - dask_ml.cluster.k_means - INFO - Starting init iteration  2/ 9 ,  1 centers
2026-06-12 10:50:24,310 - dask_ml.cluster.k_means - INFO - Finished init iteration  2/ 9 ,  1 centers in 0:00:00.072084
2026-06-12 10:50:24,337 - dask_ml.cluster.k_means - INFO - Starting init iteration  3/ 9 ,  5 centers
2026-06-12 10:50:24,406 - dask_ml.cluster.k_means - INFO - Finished init iteration  3/ 9 ,  5 centers in 0:00:00.069253
2026-06-12 10:50:24,433 - dask_ml.cluster.k_means - INFO - Starting init iteration  4/ 9 ,  7 centers
2026-06-12 10:50:24,512 - dask_ml.cluster.k_means - INFO - Finished init iteration  4/ 9 ,  7 centers in 0:00:00.077562
2026-06-12 10:50:24,536 - dask_ml.cluster.k_means - INFO - Starting init iteration  5/ 9 , 13 centers
2026-06-12 10:50:24,602 - dask_ml.cluster.k_means - INFO - Finished init iteration  5/ 9 , 13 centers in 0:00:00

Calculating cluster means and fitting priors...


In [9]:
output.data_type = 'spectral_image'
output._axes

{0: X:  X (nm) of size (2,),
 1: Y:  Y (nm) of size (2,),
 2: DC_Offset:  DC_Offset (V) of size (64,),
 3: Field:  Field (generic) of size (2,),
 4: Cycle:  Cycle (generic) of size (2,),
 5: fit_parameters:  Label (generic) of size (4,)}

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider, VBox, HBox, Output
from matplotlib.colors import Normalize

# ==============================================
# Example synthetic dataset
# ==============================================

data = output

# Axes
X_vals = output._axes[0].values
Y_vals = output._axes[1].values
DC_vals = output._axes[2].values
field_vals = output._axes[3].values
cycle_vals  = output._axes[4].values
fit_labels = ['Amplitude', 'Resonant Frequency', 'Quality Factor', 'Phase'] #need to add into the fit labels

# ==============================================
# Plotting functions
# ==============================================
# Shared state
selected_point = {'x': 0, 'y': 0}  # clicked (X,Y) index

# Output areas for dynamic updates
out_left = Output()
out_right = Output()

def plot_visualizer(dc_index, field_index, cycle_index, fit_index):
    """Main function to update the plots."""
    with out_left:
        out_left.clear_output(wait=True)
        
        # Extract 2D slice for current DC_Offset
        slice_2d = np.array(data[:, :, dc_index, field_index, cycle_index, fit_index])
        
        # Create figure
        fig, ax = plt.subplots(2, 1, figsize=(4.5, 6), 
                               gridspec_kw={'height_ratios': [4, 2]})
        
        # --- Top: Heatmap ---
        im = ax[0].imshow(slice_2d.T, origin='lower',
                          extent=[X_vals.min(), X_vals.max(), Y_vals.min(), Y_vals.max()],
                          aspect='auto', cmap='viridis',
                          norm=Normalize(vmin=np.min(slice_2d), vmax=np.max(slice_2d)))
        ax[0].set_title(f"{fit_labels[fit_index]} at DC = {DC_vals[dc_index]:.2f} V")
        ax[0].set_xlabel("X (nm)")
        ax[0].set_ylabel("Y (nm)")
        fig.colorbar(im, ax=ax[0], label=fit_labels[fit_index])
        
        # Add a marker for the selected point
        ax[0].scatter(X_vals[selected_point['x']], Y_vals[selected_point['y']], 
                      color='red', s=50, marker='x')
        
        # --- Bottom: DC waveform ---
        ax[1].plot(DC_vals, label='DC Waveform', color='black')
        ax[1].axvline(dc_index, color='red', linestyle='--')
        ax[1].set_xlabel("Index")
        ax[1].set_ylabel("Voltage (V)")
        ax[1].set_title("DC Offset Waveform")
        ax[1].legend()
        
        # Capture clicks on the heatmap
        def onclick(event):
            if event.inaxes == ax[0]:
                # Find nearest (X,Y) index
                x_idx = np.abs(X_vals - event.xdata).argmin()
                y_idx = np.abs(Y_vals - event.ydata).argmin()
                selected_point['x'] = x_idx
                selected_point['y'] = y_idx
                # Update right-hand plot
                update_right_plot(field_index, cycle_index, fit_index)
                plot_visualizer(dc_index, field_index, cycle_index, fit_index)
        
        cid = fig.canvas.mpl_connect('button_press_event', onclick)
        plt.tight_layout()
        plt.show()
    
    # Update the right plot initially
    update_right_plot(field_index, cycle_index, fit_index)

def update_right_plot(field_index, cycle_index, fit_index):
    """Plot response vs DC_Offset for selected (X,Y) point."""
    with out_right:
        out_right.clear_output(wait=True)
        
        x_idx, y_idx = selected_point['x'], selected_point['y']
        response = data[x_idx, y_idx, :, field_index, cycle_index, fit_index]
        
        plt.figure(figsize=(4, 4))
        plt.plot(DC_vals, response, marker='o', color='blue')
        plt.title(f"Response vs DC Offset at (X={X_vals[x_idx]:.1f}, Y={Y_vals[y_idx]:.1f})")
        plt.xlabel("DC Offset (V)")
        plt.ylabel(fit_labels[fit_index])
        plt.grid(True)
        plt.show()

# ==============================================
# Widgets
# ==============================================
field_dropdown = Dropdown(options=[('On-Field',0),('Off-Field',1)], value=0, description='Field:')
cycle_dropdown = Dropdown(options=[('First Cycle',0),('Second Cycle',1)], value=0, description='Cycle:')
fit_dropdown = Dropdown(options=[(name, idx) for idx, name in enumerate(fit_labels)], value=0, description='Fit:')

dc_slider = FloatSlider(value=0, min=0, max=len(DC_vals)-1,
                        step=(1), description='DC Step', continuous_update=False)

# Wrapper to convert voltage to index
def update(dc_value, field_value, cycle_value, fit_value):
    dc_index = int(dc_value)
    cycle_index = int(cycle_value) ## -1
    plot_visualizer(dc_index, field_value, cycle_index, fit_value)

# ==============================================
# Layout
# ==============================================
controls = VBox([dc_slider, field_dropdown, cycle_dropdown, fit_dropdown])

# Left side narrower, right side wider
layout = HBox([
    VBox([ out_left], layout={'width': '45%'}),
    VBox([out_right, controls], layout={'width': '55%'})
])

# Initialize visualization
update(dc_slider.value, field_dropdown.value, cycle_dropdown.value, fit_dropdown.value)

# Display
display(layout)

# Link updates
dc_slider.observe(lambda change: update(dc_slider.value, field_dropdown.value, cycle_dropdown.value, fit_dropdown.value), names='value')
field_dropdown.observe(lambda change: update(dc_slider.value, field_dropdown.value, cycle_dropdown.value, fit_dropdown.value), names='value')
cycle_dropdown.observe(lambda change: update(dc_slider.value, field_dropdown.value, cycle_dropdown.value, fit_dropdown.value), names='value')
fit_dropdown.observe(lambda change: update(dc_slider.value, field_dropdown.value, cycle_dropdown.value, fit_dropdown.value), names='value')

In [11]:
"""
Interactive per-pixel SHO fit-quality visualizer for BE-line / BEPS-slice data.
Shows: raw 2D heatmap at one frequency | raw vs fitted amplitude | raw vs fitted phase.
Click on the heatmap to pick a pixel; move the slider to change the frequency slice.
"""

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import widgets, VBox
from matplotlib.patches import Rectangle
from IPython.display import display


class InteractiveVisualizer:
    def __init__(self, raw_data, fit_data, freq_vec):
        self.raw_data = raw_data            # (X, Y, freq) complex
        self.fit_data = fit_data            # (X, Y, 4) real (Amp, w_0, Q, phi)
        self.freq_vec = freq_vec            # (freq,) real

        self.x = 0
        self.y = 0
        self.freq_slice = 0

        # Build figure with ipympl-friendly pattern: ioff() so plt.subplots
        # doesn't auto-display in a separate output area.
        plt.ioff()
        self.fig, (self.ax1, self.ax2, self.ax3) = plt.subplots(1, 3, figsize=(12, 4))
        self.fig.canvas.header_visible = False
        plt.ion()

        # Click handler on the heatmap
        self.fig.canvas.mpl_connect('button_press_event', self.on_click)

        # Slider (max is INCLUSIVE in IntSlider, so last valid index = len - 1)
        self.freq_slice_slider = widgets.IntSlider(
            min=0, max=len(freq_vec) - 1, value=0, description='Freq Slice:'
        )
        self.freq_slice_slider.observe(self._on_slider_change, names='value')

        # First draw, then display slider + figure together
        self.update_plots(self.freq_slice, self.x, self.y)
        display(VBox([self.freq_slice_slider, self.fig.canvas]))

    def _on_slider_change(self, change):
        self.update_plots(change['new'], self.x, self.y)

    def on_click(self, event):
        if event.inaxes == self.ax1 and event.xdata is not None and event.ydata is not None:
            nx, ny = self.raw_data.shape[0], self.raw_data.shape[1]
            self.x = int(np.clip(round(event.xdata), 0, nx - 1))
            self.y = int(np.clip(round(event.ydata), 0, ny - 1))
            self.update_plots(self.freq_slice, self.x, self.y)

    def update_plots(self, freq_slice, x, y):
        self.freq_slice = freq_slice

        # --- Raw 2D heatmap at this frequency slice ---
        self.ax1.clear()
        self.ax1.imshow(np.abs(self.raw_data[:, :, freq_slice]), origin='lower')
        self.ax1.set_title(f'Raw |amp| @ freq idx {freq_slice}')
        self.ax1.set_xlabel('X')
        self.ax1.set_ylabel('Y')
        self.ax1.add_patch(Rectangle(
            (x - 0.5, y - 0.5), 1, 1, linewidth=2, edgecolor='red', facecolor='none'
        ))

        # --- Raw vs fitted amplitude/phase at selected pixel ---
        amp_data = np.abs(self.raw_data[x, y, :])
        phase_data = np.angle(self.raw_data[x, y, :])

        fit_parms = self.fit_data[x, y, :]
        sho_fit = SHO_fit_flattened(self.freq_vec, *fit_parms)
        n = len(sho_fit) // 2
        sho_fit_complex = sho_fit[:n] + 1j * sho_fit[n:]
        amp_fit = np.abs(sho_fit_complex)
        phase_fit = np.angle(sho_fit_complex)

        self.ax2.clear()
        self.ax2.plot(self.freq_vec, amp_data, 'ro', label='Raw')
        self.ax2.plot(self.freq_vec, amp_fit, 'r-', label='Fit')
        self.ax2.set_title(f'Amplitude @ ({x},{y})')
        self.ax2.set_xlabel('Frequency (Hz)')
        self.ax2.set_ylabel('Amplitude (a.u.)')
        self.ax2.legend()

        self.ax3.clear()
        self.ax3.plot(self.freq_vec, phase_data, 'ro', label='Raw')
        self.ax3.plot(self.freq_vec, phase_fit, 'r-', label='Fit')
        self.ax3.set_title(f'Phase @ ({x},{y})')
        self.ax3.set_xlabel('Frequency (Hz)')
        self.ax3.set_ylabel('Phase (rad)')
        self.ax3.legend()

        try:
            self.fig.canvas.draw_idle()
        except AttributeError:
            pass

# Example usage
# InteractiveVisualizer is BE-line-only (expects 3D inputs).
# For BEPS data, pick one (DC, field, cycle) slice.
raw_data = np.array(beps_small[:, :, :, 0, 0, 0])  # (X, Y, freq) at DC=0, field=0, cycle=0
fit_data = np.array(output[:, :, 0, 0, 0, :])      # (X, Y, 4 params) same slice

# Instantiate the visualizer
visualizer = InteractiveVisualizer(raw_data, fit_data, freq_vec)